In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog", "consumer_goods", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
base_path = f'/Volumes/consumer_goods/files/input_data_files/child_company/{data_source}/*.csv'

print(catalog)
print(data_source)
print(base_path)


consumer_goods
gross_price
/Volumes/consumer_goods/files/input_data_files/child_company/gross_price/*.csv


In [0]:
df = (spark.read.format("csv")
        .option("header", "True")
        .option("inferSchema", "True")
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*","_metadata.file_name")
)
df.show(10)

+----------+----------+-----------+--------------------+---------------+
|product_id|     month|gross_price|      read_timestamp|      file_name|
+----------+----------+-----------+--------------------+---------------+
|  25891101|2025/07/01|        -84|2026-03-03 11:41:...|gross_price.csv|
|  25891101|01/08/2025|    unknown|2026-03-03 11:41:...|gross_price.csv|
|  25891101|2025/09/01|         84|2026-03-03 11:41:...|gross_price.csv|
|  25891101|2025-10-01|         83|2026-03-03 11:41:...|gross_price.csv|
|  25891101|2025-11-01|         83|2026-03-03 11:41:...|gross_price.csv|
|  88888888|2025-12-01|        -83|2026-03-03 11:41:...|gross_price.csv|
|  25891102|2025-07-01|         68|2026-03-03 11:41:...|gross_price.csv|
|  25891102|2025-08-01|         68|2026-03-03 11:41:...|gross_price.csv|
|  25891102|2025-09-01|         68|2026-03-03 11:41:...|gross_price.csv|
|  25891102|2025-10-01|         69|2026-03-03 11:41:...|gross_price.csv|
+----------+----------+-----------+----------------

In [0]:
df.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("overwrite").saveAsTable(f"{catalog}.bronze.{data_source}")

Month is having different formats

In [0]:
df.select("month").distinct().show()

+----------+
|     month|
+----------+
|2025-11-01|
|2025/09/01|
|2025-08-01|
|01/08/2025|
|01-09-2025|
|01-08-2025|
|2025/11/01|
|2025/08/01|
|01/12/2025|
|2025-10-01|
|2025/10/01|
|2025/07/01|
|01-12-2025|
|01/10/2025|
|01/09/2025|
|2025-12-01|
|2025-07-01|
|2025-09-01|
+----------+



trying to read with different formats

In [0]:
df_formatted = df.withColumn("month",
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy")
    )
)
df_formatted.show(10)

+----------+----------+-----------+--------------------+---------------+
|product_id|     month|gross_price|      read_timestamp|      file_name|
+----------+----------+-----------+--------------------+---------------+
|  25891101|2025-07-01|        -84|2026-03-03 11:42:...|gross_price.csv|
|  25891101|2025-08-01|    unknown|2026-03-03 11:42:...|gross_price.csv|
|  25891101|2025-09-01|         84|2026-03-03 11:42:...|gross_price.csv|
|  25891101|2025-10-01|         83|2026-03-03 11:42:...|gross_price.csv|
|  25891101|2025-11-01|         83|2026-03-03 11:42:...|gross_price.csv|
|  88888888|2025-12-01|        -83|2026-03-03 11:42:...|gross_price.csv|
|  25891102|2025-07-01|         68|2026-03-03 11:42:...|gross_price.csv|
|  25891102|2025-08-01|         68|2026-03-03 11:42:...|gross_price.csv|
|  25891102|2025-09-01|         68|2026-03-03 11:42:...|gross_price.csv|
|  25891102|2025-10-01|         69|2026-03-03 11:42:...|gross_price.csv|
+----------+----------+-----------+----------------

Gross price value fixing
- Checking for negative value
- if yes, multiply with -1
- else 0



In [0]:
df_formatted = df_formatted.withColumn("gross_price",
    F.when(F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'), 
           F.when(F.col("gross_price").cast("double") < 0, -1 * F.col("gross_price").cast("double"))
            .otherwise(F.col("gross_price").cast("double")))
    .otherwise(0)
)
df_formatted.show(10)

+----------+----------+-----------+--------------------+---------------+
|product_id|     month|gross_price|      read_timestamp|      file_name|
+----------+----------+-----------+--------------------+---------------+
|  25891101|2025-07-01|       84.0|2026-03-03 11:42:...|gross_price.csv|
|  25891101|2025-08-01|        0.0|2026-03-03 11:42:...|gross_price.csv|
|  25891101|2025-09-01|       84.0|2026-03-03 11:42:...|gross_price.csv|
|  25891101|2025-10-01|       83.0|2026-03-03 11:42:...|gross_price.csv|
|  25891101|2025-11-01|       83.0|2026-03-03 11:42:...|gross_price.csv|
|  88888888|2025-12-01|       83.0|2026-03-03 11:42:...|gross_price.csv|
|  25891102|2025-07-01|       68.0|2026-03-03 11:42:...|gross_price.csv|
|  25891102|2025-08-01|       68.0|2026-03-03 11:42:...|gross_price.csv|
|  25891102|2025-09-01|       68.0|2026-03-03 11:42:...|gross_price.csv|
|  25891102|2025-10-01|       69.0|2026-03-03 11:42:...|gross_price.csv|
+----------+----------+-----------+----------------

We need product id from products table

In [0]:
df_products = spark.table("consumer_goods.silver.products") 
df_joined = df_formatted.join(df_products.select("product_id", "product_code"), on="product_id", how="inner")
df_joined = df_joined.select("product_id", "product_code", "month", "gross_price", "read_timestamp", "file_name")

df_joined.show(5)

+----------+--------------------+----------+-----------+--------------------+---------------+
|product_id|        product_code|     month|gross_price|      read_timestamp|      file_name|
+----------+--------------------+----------+-----------+--------------------+---------------+
|  25891503|062f5574bbdf4386b...|2025-12-01|      281.0|2026-03-03 11:46:...|gross_price.csv|
|  25891102|e92c739a8d78cd6cb...|2025-12-01|       69.0|2026-03-03 11:46:...|gross_price.csv|
|  25891302|d9ebd1ca64d23951a...|2025-12-01|      300.0|2026-03-03 11:46:...|gross_price.csv|
|  25891403|77b6f538a9d0e0cf8...|2025-12-01|       50.0|2026-03-03 11:46:...|gross_price.csv|
|  25891301|3cab59f0592428527...|2025-12-01|      493.0|2026-03-03 11:46:...|gross_price.csv|
+----------+--------------------+----------+-----------+--------------------+---------------+
only showing top 5 rows


In [0]:
df_joined.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true")\
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.silver.{data_source}")

In [0]:
df_gold = df_joined.select("product_code", "month", "gross_price")
df_gold.show(5)

+--------------------+----------+-----------+
|        product_code|     month|gross_price|
+--------------------+----------+-----------+
|062f5574bbdf4386b...|2025-12-01|      281.0|
|e92c739a8d78cd6cb...|2025-12-01|       69.0|
|d9ebd1ca64d23951a...|2025-12-01|      300.0|
|77b6f538a9d0e0cf8...|2025-12-01|       50.0|
|3cab59f0592428527...|2025-12-01|      493.0|
+--------------------+----------+-----------+
only showing top 5 rows


In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.gold.sb_dim_{data_source}")

Merge with parent by aggregating

In [0]:
from pyspark.sql.window import Window
df_gold_agg = (
    df_gold
    .withColumn("year", F.year("month"))
    # 0 = non-zero price, 1 = zero price  ➜ non-zero comes first
    .withColumn("is_zero", F.when(F.col("gross_price") == 0, 1).otherwise(0))
)

w = (
    Window
    .partitionBy("product_code", "year")
    .orderBy(F.col("is_zero"), F.col("month").desc())
)


df_gold_latest_price = (
    df_gold_agg
      .withColumn("rnk", F.row_number().over(w))
      .filter(F.col("rnk") == 1)
)
df_gold_latest_price.show(5)

+--------------------+----------+-----------+----+-------+---+
|        product_code|     month|gross_price|year|is_zero|rnk|
+--------------------+----------+-----------+----+-------+---+
|062f5574bbdf4386b...|2025-12-01|      281.0|2025|      0|  1|
|0cb7b2f42657b625f...|2025-10-01|      100.0|2025|      0|  1|
|102628255d24304d6...|2025-12-01|       86.0|2025|      0|  1|
|2e387cef1424d6e7b...|2025-12-01|      108.0|2025|      0|  1|
|3cab59f0592428527...|2025-12-01|      493.0|2025|      0|  1|
+--------------------+----------+-----------+----+-------+---+
only showing top 5 rows


In [0]:
df_gold_latest_price = df_gold_latest_price.select("product_code", "year", "gross_price").withColumnRenamed("gross_price", "price_inr").select("product_code", "price_inr", "year")


In [0]:
df_gold_latest_price = df_gold_latest_price.withColumn("year", F.col("year").cast("string"))
df_gold_latest_price.show(5)

+--------------------+---------+----+
|        product_code|price_inr|year|
+--------------------+---------+----+
|062f5574bbdf4386b...|    281.0|2025|
|0cb7b2f42657b625f...|    100.0|2025|
|102628255d24304d6...|     86.0|2025|
|2e387cef1424d6e7b...|    108.0|2025|
|3cab59f0592428527...|    493.0|2025|
+--------------------+---------+----+
only showing top 5 rows


In [0]:
delta_table = DeltaTable.forName(spark, "consumer_goods.gold.dim_gross_price")


delta_table.alias("target").merge(
    source=df_gold_latest_price.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
DESCRIBE HISTORY consumer_goods.gold.dim_gross_price

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-03-03T11:55:59.000Z,71320613334569,yvishnu224@gmail.com,MERGE,"Map(predicate -> [""(product_code#14889 = product_code#14829)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2662572372511534),4c28bd41-b2d9-4aef-a3be-812593398254,0303-112818-wxc4dad7-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1987, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3545, materializeSourceTimeMs -> 779, numTargetRowsInserted -> 17, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1565, numTargetRowsUpdated -> 0, numOutputRows -> 17, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 17, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1096)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-02T11:41:59.000Z,71320613334569,yvishnu224@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(903424024329876),5b8001f9-031b-46be-a945-fd404fc13af6,0302-113816-lp5wn68f-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 6236, numDeletionVectorsRemoved -> 0, numOutputRows -> 794, numOutputBytes -> 6236)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-02-28T06:38:03.000Z,71320613334569,yvishnu224@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(903424024329876),159fc71a-dbae-440c-9dd5-6270ec4f25a7,0228-052951-nxc2v7k3-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 794, numOutputBytes -> 6236)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
